# Skintelligent Recommendation System

In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# loading dataset
full_data = pd.read_csv('../data/reviews_all_products.csv')
full_data.head()

/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_87397/3277208589.py:2: DtypeWarning: Columns (1,28) have mixed types. Specify dtype option on import or set low_memory=False.
  full_data = pd.read_csv('../data/reviews_all_products.csv')


,Unnamed: 0,author_id,review_rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,...,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0


In [3]:
full_data.shape

(975094, 45)

In [4]:
print(full_data.columns.tolist())

['Unnamed: 0', 'author_id', 'review_rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'review_price', 'product_name_y', 'brand_id', 'brand_name_y', 'loves_count', 'product_rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'product_price', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']


### Feature Engineering
---

In [5]:
# helpfullness weighted rating
full_data['helpfulness'].fillna(0, inplace=True)

full_data['weighted_rating'] = (
    full_data['review_rating'] *
    (1 + full_data['total_pos_feedback_count'])
)

In [6]:
full_data[['weighted_rating', 'helpfulness', 'review_rating', 'total_pos_feedback_count']].head()

,weighted_rating,helpfulness,review_rating,total_pos_feedback_count
0,15,1.0,5,2
1,1,0.0,1,0
2,5,0.0,5,0
3,5,0.0,5,0
4,5,0.0,5,0


**Colummn to find how popular this product is among people with your skin type**

In [7]:
# group by product_id and skin_type to get count of each skin type for each product
skin_match = full_data.groupby(['product_id', 'skin_type']).size().unstack(fill_value=0)

# convert counts to ratios
skin_match_ratio = skin_match.div(skin_match.sum(axis=1), axis=0)

### Product Level Aggregation
---

In [8]:
product_stats = full_data.groupby('product_id').agg({
    'product_name': 'first',
    'brand_name': 'first',
    'product_rating': 'mean',
    'review_rating': ['mean', 'std', 'count'],
    'weighted_rating': 'mean',
    'loves_count': 'mean',
    'reviews': 'mean',
    'ingredients': 'first',
    'primary_category': 'first',
    'secondary_category': 'first',
    'tertiary_category': 'first',
    'highlights': 'first',
    'product_price': 'mean',
    'sephora_exclusive': 'mean',
    'new': 'mean'
}).reset_index()

product_stats.columns = [
    'product_id','product_name','brand_name','product_rating',
    'avg_review_rating','rating_std','review_count',
    'weighted_rating','loves_count','review_total',
    'ingredients','primary_category','secondary_category',
    'tertiary_category','highlights','price','exclusive','is_new'
]

### Core Scoring Features
---

#### Popularity

In [9]:
product_stats['popularity'] = np.log1p(
    product_stats['loves_count'] + product_stats['review_count']
)

product_stats['popularity_score'] = (
    (product_stats['popularity'] - product_stats['popularity'].min()) /
    (product_stats['popularity'].max() - product_stats['popularity'].min())
)

#### Rating (Confidence Weighted)

In [10]:
product_stats['confidence'] = (
    np.log1p(product_stats['review_count']) /
    np.log1p(product_stats['review_count'].max())
)

product_stats['rating_score'] = (product_stats['avg_review_rating'] - 1) / 4

product_stats['adjusted_rating'] = (
    product_stats['rating_score'] * product_stats['confidence']
)

#### "Worth The Hype" Score

In [11]:
product_stats['rating_std'] = product_stats['rating_std'].fillna(0)
product_stats['hype_score'] = 1 / (1 + product_stats['rating_std'])

### Ingredient Model
---

In [12]:
ingredient_risk = pd.read_csv('../data/ingredient_risk_table.csv')

In [13]:
# consistent comparisons
def normalize_string(s):
    if pd.isna(s):
        return ""
    return str(s).strip().lower().replace(" ", "").replace("-", "")

# ingredient compatibility function
def ingredient_compatibility_score(ingredient_string, user_profile, ingredient_risk):
    """
    Compute a compatibility score (0-1) based on ingredient risks and user profile.
    """
    if pd.isna(ingredient_string) or not ingredient_string:
        return 0.5  # neutral score if no ingredients listed

    # Normalize ingredients
    ingredients = [ing.strip().lower() for ing in ingredient_string.split(",")]

    score = 1.0
    penalties = 0

    # Risk weights
    risk_weight = {
        "acne_risk": 0.2,
        "irritation_risk": 0.25,
        "dry_skin_risk": 0.15,
        "sensitivity_risk": 0.25
    }

    # Apply risk-based penalties
    for _, row in ingredient_risk.iterrows():
        ing_name = row["ingredient_name"].lower().strip()
        if ing_name in ingredients:

            # Acne-prone
            if user_profile.get("acne_prone", False):
                penalties += row["acne_risk"] * risk_weight["acne_risk"]

            # Sensitive skin
            if user_profile.get("skin_sensitivity", "Low") in ["High", "Very High"]:
                penalties += row["irritation_risk"] * risk_weight["irritation_risk"]

            # Dry skin
            if user_profile.get("skin_type", "").lower() == "dry":
                penalties += row["dry_skin_risk"] * risk_weight["dry_skin_risk"]

    # Fragrance allergy
    if user_profile.get("fragrance_allergy", False) and "fragrance" in ingredients:
        penalties += 0.3

    final_score = score - penalties
    return max(min(final_score, 1), 0)

In [14]:
# ingredient reason codes function
def ingredient_reason_codes(ingredient_string, user_profile, ingredient_risk):
    """
    Generate reason codes explaining ingredient risks for the user.
    """
    reasons = []
    if pd.isna(ingredient_string) or not ingredient_string:
        return reasons

    ingredients = [ing.strip().lower() for ing in ingredient_string.split(",")]

    for _, row in ingredient_risk.iterrows():
        ing_name = row["ingredient_name"].lower().strip()
        if ing_name in ingredients:

            # Acne-prone
            if user_profile.get("acne_prone", False) and row["acne_risk"] > 0:
                reasons.append(f"Ingredient {row['ingredient_name']} may trigger acne")

            # Sensitive skin
            if user_profile.get("skin_sensitivity", "Low") in ["High", "Very High"] and row["irritation_risk"] > 0:
                reasons.append(f"Ingredient {row['ingredient_name']} may irritate sensitive skin")

            # Dry skin
            if user_profile.get("skin_type", "").lower() == "dry" and row["dry_skin_risk"] > 0:
                reasons.append(f"Ingredient {row['ingredient_name']} may worsen dryness")

    # Fragrance allergy
    if user_profile.get("fragrance_allergy", False) and "fragrance" in ingredients:
        reasons.append("Contains fragrance (allergy warning)")

    return reasons

### Collaborative Filtering (Similar Users Algorithm)
---

In [ ]:
# need to work on this function more - just a placeholder for now

### Final Recommender System
---

In [15]:
def recommend(user_profile, user_id=None, top_n=10):
    
    df = product_stats.copy()
    
    # ingredient score
    df['ingredient_score'] = df['ingredients'].apply(
        lambda x: ingredient_score(x, user_profile)
    )
    
    # skin match
    if user_profile['skin_type'] in skin_match_ratio.columns:
        df['skin_match'] = df['product_id'].map(
            skin_match_ratio[user_profile['skin_type']]
        ).fillna(0)
    else:
        df['skin_match'] = 0.5
    
    # NLP placeholder - CHANGE LATER - KRISTEN'S PART
    df['sentiment'] = 0.5
    
    # collaborative filtering - CHANGE LATER - EMILY'S PART
    if user_id is not None:
        collab = get_collab_scores(user_id)
        df['collab'] = df['product_id'].map(collab).fillna(0)
    else:
        df['collab'] = 0
    
    # normalize collab
    if df['collab'].max() > 0:
        df['collab'] = df['collab'] / df['collab'].max()
    
    # FINAL SCORE (FULL HYBRID)
    df['final_score'] = (
        0.25 * df['ingredient_score'] +
        0.15 * df['adjusted_rating'] +
        0.10 * df['popularity_score'] +
        0.10 * df['hype_score'] +
        0.15 * df['skin_match'] +
        0.10 * df['price_score'] +
        0.15 * df['collab']
    )
    
    return df.sort_values('final_score', ascending=False).head(top_n)